# Course Outcome 1 (CO1) - Online Shopping Cart System

## 1. Problem Statement
Design a Python-based online shopping cart system using procedural programming (functions and standard dictionaries). The system must allow users to view products, add products, change quantities, remove products, apply compound discounts (percentage + flat discount), calculate the subtotal, calculate category-specific GST, and compute the final bill with a formatted receipt.

## 2. Divide into Parts/Modules
To solve this problem cleanly, the system is divided into the following functional parts:
1. **Core Operations Logic**: Functions to modify the shopping cart data (`add_product`, `remove_product`, `change_quantity`).
2. **Computational Logic**: Functions to calculate subtotal, compound discounts, and category-wise GST.
3. **Display Logic**: A function to render the final checkout receipt.
4. **Data Store & Input Setup**: Defines product inventory data, category tax rates, and default parameters.
5. **Sample Edge Cases (Test Cases)**: Conceptual descriptions of tests to verify calculations under extreme conditions.
6. **Driver Logic (Main)**: The interactive console menu loop that accepts user choices and drives shopping operations.

## 3. Abstraction
Abstraction helps simplify the problem by focusing only on what is necessary and ignoring irrelevant details.

### Needed Details (Essential Information):
* **Inventory Database**: Mapping of product ID to name, unit price, and category.
* **Cart Contents**: Mapping of product ID to purchase quantity.
* **Discount Rules**: Percentage rate and flat discount amount.
* **GST Rates**: Category-specific GST rate percentages.

### Useless Details (Irrelevant Information):
* **Store Attributes**: Physical counter design, store floor layouts, or employee rosters.
* **Payment Details**: Specific payment gateway, validation methods, or merchant IDs.
* **Customer Info**: Customer phone number, physical address, or membership tier.

## 4. Algorithm
The shopping cart operations are performed as follows:
1. Initialize empty cart, inventory catalog, and category-wise tax rates.
2. Loop to display options (1: View Products, 2: View Cart, 3: Add Product, 4: Change Qty, 5: Remove, 6: Apply Discount, 7: Print Invoice, 8: Exit).
3. On Product Add/Modify, validate quantity ($> 0$). On remove, validate existence in cart.
4. Compute subtotal as $\sum (\text{Price}_i \times \text{Quantity}_i)$.
5. Compute discount: apply percentage discount first, then add flat discount. Cap discount at subtotal value.
6. Distribute discount proportionally across items: $\text{ratio} = \frac{\text{Subtotal} - \text{Discount}}{\text{Subtotal}}$.
7. Compute GST: for each item, calculate tax on its discounted net price using its category rate.
8. Compute final total: $\text{Subtotal} - \text{Discount} + \text{GST}$.

## 5. Core Logic

In [1]:
def add_product(cart, product_id, quantity):
    if quantity <= 0:
        raise ValueError()
    cart[product_id] = cart.get(product_id, 0) + quantity

def remove_product(cart, product_id):
    if product_id not in cart:
        raise KeyError()
    del cart[product_id]

def change_quantity(cart, product_id, quantity):
    if product_id not in cart:
        raise KeyError()
    if quantity <= 0:
        raise ValueError()
    cart[product_id] = quantity

def calculate_subtotal(cart, inventory):
    subtotal = 0.0
    for pid, qty in cart.items():
        subtotal += inventory[pid]["price"] * qty
    return subtotal

def calculate_discount(subtotal, discount_percent, discount_flat):
    pct_disc = subtotal * (discount_percent / 100.0)
    total_disc = pct_disc + discount_flat
    return min(total_disc, subtotal)

def calculate_gst(cart, inventory, discount_ratio, gst_rates, default_rate=0.18):
    total_gst = 0.0
    for pid, qty in cart.items():
        item_sub = inventory[pid]["price"] * qty
        item_disc = item_sub * discount_ratio
        category = inventory[pid]["category"]
        rate = gst_rates.get(category, default_rate)
        total_gst += item_disc * rate
    return total_gst

def print_receipt(cart, inventory, discount_percent, discount_flat, gst_rates, default_rate=0.18):
    subtotal = calculate_subtotal(cart, inventory)
    disc_val = calculate_discount(subtotal, discount_percent, discount_flat)
    discount_ratio = (subtotal - disc_val) / subtotal if subtotal > 0 else 0.0
    gst_val = calculate_gst(cart, inventory, discount_ratio, gst_rates, default_rate)
    grand_total = subtotal - disc_val + gst_val
    
    print("=" * 75)
    print("                   ONLINE SHOPPING CART INVOICE                    ")
    print("=" * 75)
    print(f"{'ID':<6} {'Product Name':<20} {'Qty':<4} {'Price':<8} {'Subtotal':<10} {'GST':<8} {'Total':<10}")
    print("-" * 75)
    
    for pid, qty in cart.items():
        price = inventory[pid]["price"]
        item_sub = price * qty
        item_net = item_sub * discount_ratio
        rate = gst_rates.get(inventory[pid]["category"], default_rate)
        item_gst = item_net * rate
        item_tot = item_net + item_gst
        name = inventory[pid]["name"]
        print(f"{pid:<6} {name:<20} {qty:<4} ${price:<7.2f} ${item_sub:<9.2f} ${item_gst:<7.2f} ${item_tot:<9.2f}")
        
    print("-" * 75)
    print(f"{'Subtotal:':<63} ${subtotal:.2f}")
    if disc_val > 0:
        disc_label = f"Discount ({discount_percent}% off + ${discount_flat:.2f} flat):"
        print(f"{disc_label:<63} -${disc_val:.2f}")
    print(f"{'GST Total:':<63} ${gst_val:.2f}")
    print("-" * 75)
    print(f"{'GRAND TOTAL:':<63} ${grand_total:.2f}")
    print("=" * 75)
    print("                 Thank you for shopping with us!                 ")
    print("=" * 75)


## 6. Data Store & Input Setup

In [2]:
inventory = {
    "P001": {"name": "Laptop", "price": 1200.00, "category": "Electronics"},
    "P002": {"name": "Headphones", "price": 150.00, "category": "Electronics"},
    "P003": {"name": "Winter Jacket", "price": 80.00, "category": "Clothing"},
    "P004": {"name": "Algorithmic Book", "price": 45.00, "category": "Books"},
    "P005": {"name": "Organic Apples", "price": 12.00, "category": "Groceries"}
}

gst_rates = {
    "Electronics": 0.18,
    "Clothing": 0.12,
    "Groceries": 0.05,
    "Books": 0.00
}

cart = {}
discount_percent = 10.0
discount_flat = 15.0


## 7. Sample Edge Cases (Test Cases)
Here are the test scenarios designed to verify system correctness:

### Test Case 1: Standard Add and Modify
* **Input**:
  * Add `P001` (Laptop, qty 1), Add `P002` (Headphones, qty 2)
  * Change quantity of `P002` to 1
* **Expected Output**: Cart contains `P001` (qty 1) and `P002` (qty 1)

### Test Case 2: Product Removal
* **Input**:
  * Remove `P002` from cart
* **Expected Output**: Cart contains only `P001`

### Test Case 3: Invalid Quantity Bound
* **Input**:
  * Try `change_quantity` of `P001` to 0 or negative value
* **Expected Output**: Raises `ValueError` exception

### Test Case 4: Category-Wise GST Calculation
* **Input**:
  * Laptop ($1200, Electronics category at 18% GST)
* **Expected Output**: GST amount is $216.00

### Test Case 5: Discount Overflow Cap
* **Input**:
  * Subtotal: $120. Flat discount: $200.
* **Expected Output**: Applied discount capped at $120 (total bill before tax cannot drop below 0)

## 8. Driver Logic (Main)

In [3]:
import builtins

simulated_inputs = [
    "1",
    "3", "P001", "1",
    "3", "P002", "2",
    "3", "P004", "1",
    "3", "P005", "3",
    "4", "P002", "1",
    "4", "P005", "5",
    "5", "P004",
    "2",
    "6", "10", "15",
    "7",
    "8"
]

def mock_input(prompt=""):
    if not simulated_inputs:
        return "8"
    val = simulated_inputs.pop(0)
    print(f"{prompt}{val}")
    return val

builtins.input = mock_input

cart = {}
discount_percent = 0.0
discount_flat = 0.0

while True:
    print("\n--- ONLINE SHOPPING MENU ---")
    print("1. View Products")
    print("2. View Cart")
    print("3. Add Product")
    print("4. Change Quantity")
    print("5. Remove Product")
    print("6. Apply Discount")
    print("7. Print Receipt")
    print("8. Exit")
    choice = input("Enter choice (1-8): ")
    if choice == "1":
        print("-" * 45)
        print(f"{'ID':<6} {'Product Name':<20} {'Price':<10} {'Category':<10}")
        print("-" * 45)
        for pid, info in inventory.items():
            print(f"{pid:<6} {info['name']:<20} ${info['price']:<9.2f} {info['category']}")
        print("-" * 45)
    elif choice == "2":
        if not cart:
            print("Cart is empty.")
        else:
            print("-" * 45)
            print(f"{'ID':<6} {'Product Name':<20} {'Qty':<6} {'Price':<10}")
            print("-" * 45)
            for pid, qty in cart.items():
                print(f"{pid:<6} {inventory[pid]['name']:<20} {qty:<6} ${inventory[pid]['price']:<9.2f}")
            print("-" * 45)
    elif choice == "3":
        pid = input("Enter Product ID: ")
        if pid not in inventory:
            print("Invalid Product ID.")
            continue
        try:
            qty = int(input("Enter Quantity: "))
            add_product(cart, pid, qty)
            print("Product added.")
        except ValueError:
            print("Invalid quantity.")
    elif choice == "4":
        pid = input("Enter Product ID: ")
        if pid not in cart:
            print("Product not in cart.")
            continue
        try:
            qty = int(input("Enter Quantity: "))
            change_quantity(cart, pid, qty)
            print("Quantity updated.")
        except ValueError:
            print("Invalid quantity.")
    elif choice == "5":
        pid = input("Enter Product ID: ")
        try:
            remove_product(cart, pid)
            print("Product removed.")
        except KeyError:
            print("Product not in cart.")
    elif choice == "6":
        try:
            discount_percent = float(input("Enter discount percent (0-100): "))
            discount_flat = float(input("Enter flat discount amount: "))
            if not (0 <= discount_percent <= 100) or discount_flat < 0:
                raise ValueError()
            print("Discounts applied.")
        except ValueError:
            print("Invalid discount values.")
    elif choice == "7":
        if not cart:
            print("Cart is empty.")
        else:
            print_receipt(cart, inventory, discount_percent, discount_flat, gst_rates)
    elif choice == "8":
        break
    else:
        print("Invalid choice.")



--- ONLINE SHOPPING MENU ---
1. View Products
2. View Cart
3. Add Product
4. Change Quantity
5. Remove Product
6. Apply Discount
7. Print Receipt
8. Exit
Enter choice (1-8): 1
---------------------------------------------
ID     Product Name         Price      Category  
---------------------------------------------
P001   Laptop               $1200.00   Electronics
P002   Headphones           $150.00    Electronics
P003   Winter Jacket        $80.00     Clothing
P004   Algorithmic Book     $45.00     Books
P005   Organic Apples       $12.00     Groceries
---------------------------------------------

--- ONLINE SHOPPING MENU ---
1. View Products
2. View Cart
3. Add Product
4. Change Quantity
5. Remove Product
6. Apply Discount
7. Print Receipt
8. Exit
Enter choice (1-8): 3
Enter Product ID: P001
Enter Quantity: 1
Product added.

--- ONLINE SHOPPING MENU ---
1. View Products
2. View Cart
3. Add Product
4. Change Quantity
5. Remove Product
6. Apply Discount
7. Print Receipt
8. Exit
Ent